# 🎓 Cloud-Based Digital Certificate Generator
This notebook runs the Certificate Generator web application. It automatically mounts your Google Drive, installs dependencies, and launches the Gradio UI.

In [ ]:
# Install required dependencies
!pip install gradio pillow reportlab

# Install fonts for the certificate (Linux default for Colab)
!apt-get install fonts-liberation

In [ ]:
import os
import gradio as gr
from PIL import Image, ImageDraw, ImageFont
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import landscape, A4
from google.colab import drive

# 1. Cloud Storage: Mount Google Drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/Generated_Certificates'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"\n📁 Certificates will be saved to your Google Drive at: {SAVE_DIR}")

# 2. Template Generation
TEMPLATE_PATH = "certificate_template.png"

def create_default_template():
    if not os.path.exists(TEMPLATE_PATH):
        img = Image.new('RGB', (1200, 800), color=(255, 255, 255))
        draw = ImageDraw.Draw(img)
        draw.rectangle([20, 20, 1180, 780], outline="#D4AF37", width=10) # Gold
        draw.rectangle([40, 40, 1160, 760], outline="#4B0082", width=5)  # Purple
        img.save(TEMPLATE_PATH)

create_default_template()

# 3. Certificate Generation Logic
def generate_certificate(student_name, event_name, organizer_name, event_date):
    if not student_name or not event_name or not organizer_name or not event_date:
        return None, None, "❌ Error: All fields are required."
    
    try:
        img = Image.open(TEMPLATE_PATH)
        draw = ImageDraw.Draw(img)
        
        # Load Fonts
        try:
            font_title = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf", 60)
            font_text = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf", 40)
            font_name = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf", 80)
        except IOError:
            font_title = ImageFont.load_default()
            font_text = ImageFont.load_default()
            font_name = ImageFont.load_default()

        def draw_centered_text(text, y, font, fill):
            if hasattr(draw, 'textbbox'):
                bbox = draw.textbbox((0, 0), text, font=font)
                w = bbox[2] - bbox[0]
            else:
                w, h = draw.textsize(text, font=font)
            draw.text(((1200 - w) / 2, y), text, font=font, fill=fill)

        draw_centered_text("CERTIFICATE OF PARTICIPATION", 150, font_title, "#4B0082")
        draw_centered_text("This is proudly presented to", 250, font_text, "#000000")
        draw_centered_text(student_name.upper(), 350, font_name, "#D4AF37")
        draw_centered_text("for active participation in the event", 480, font_text, "#000000")
        draw_centered_text(event_name, 550, font_title, "#4B0082")
        draw_centered_text(f"Organized by {organizer_name} on {event_date}", 680, font_text, "#000000")

        # Save Preview Image
        safe_name = student_name.replace(" ", "_").replace("/", "")
        img_filename = f"{safe_name}_certificate.png"
        img_path = os.path.join(SAVE_DIR, img_filename)
        img.save(img_path)
        
        # Save PDF
        pdf_filename = f"{safe_name}_certificate.pdf"
        pdf_path = os.path.join(SAVE_DIR, pdf_filename)
        c = canvas.Canvas(pdf_path, pagesize=landscape(A4))
        c.drawImage(img_path, 0, 0, width=842, height=595)
        c.save()
        
        return img_path, pdf_path, f"✅ Success! Saved to {SAVE_DIR}"
    except Exception as e:
        return None, None, f"❌ Error generating certificate: {str(e)}"

def reset_fields():
    return "", "", "", "", None, None, "Fields reset."

# 4. Gradio Interface (Frontend)
css = """
.container { max-width: 900px; margin: auto; }
h1 { text-align: center; color: #4B0082; }
.generate-btn { background-color: #4B0082 !important; color: white !important; font-weight: bold !important; }
.reset-btn { background-color: #D4AF37 !important; color: white !important; font-weight: bold !important; }
"""

with gr.Blocks(css=css, theme=gr.themes.Default(primary_hue="purple", secondary_hue="yellow")) as demo:
    gr.Markdown("# 🎓 Cloud Certificate Generator")
    gr.Markdown("<p style='text-align: center;'>Generate professional digital certificates automatically and store them securely in the cloud (Google Drive).</p>")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📝 Enter Details")
            student_name = gr.Textbox(label="Student Name", placeholder="e.g. John Doe")
            event_name = gr.Textbox(label="Event Name", placeholder="e.g. Cloud Computing Workshop")
            organizer_name = gr.Textbox(label="Organizer Name", placeholder="e.g. Tech Club")
            event_date = gr.Textbox(label="Event Date", placeholder="e.g. October 15, 2026")
            
            with gr.Row():
                generate_btn = gr.Button("✨ Generate Certificate", elem_classes="generate-btn")
                reset_btn = gr.Button("🔄 Reset", elem_classes="reset-btn")
            
            status_message = gr.Textbox(label="Status", interactive=False)
            
        with gr.Column(scale=1):
            gr.Markdown("### 👁️ Preview & Download")
            preview_image = gr.Image(label="Certificate Preview", type="filepath")
            download_pdf = gr.File(label="Download PDF Certificate")
            
    generate_btn.click(
        fn=generate_certificate,
        inputs=[student_name, event_name, organizer_name, event_date],
        outputs=[preview_image, download_pdf, status_message]
    )
    
    reset_btn.click(
        fn=reset_fields,
        inputs=[],
        outputs=[student_name, event_name, organizer_name, event_date, preview_image, download_pdf, status_message]
    )

# 5. Launch App
demo.launch(debug=True, share=True)
